# Metryki zamiast zgadywania

## Cel: Zrozumiesz, dlaczego publiczne rankingi wprowadzają w błąd i zbudujesz fundament własnego systemu oceniania, który da Ci inżynierską pewność przy wdrażaniu zmian.

### Czego się nauczysz:
1. dlaczego opieranie się na intuicji przy testowaniu złożonych systemów prowadzi do błędów produkcyjnych
2. w jaki sposób giganci technologiczni manipulują publicznymi rankingami poprzez selekcję wyników
3. na czym polega zjawisko skażenia danych i dlaczego modele znają pytania egzaminacyjne na pamięć
4. dlaczego popularne benchmarki często promują styl wypowiedzi kosztem faktycznej logiki i wydajności
5. dlaczego jedyną drogą do stabilnego produktu jest stworzenie własnego, prywatnego zestawu testów
6. jak wykorzystać model językowy w roli bezstronnego sędziego do automatyzacji procesu sprawdzania
7. dlaczego w inżynierii systemów agentowych binarne oceny są lepsze od subiektywnych skal punktowych


> **"If you can't measure it, you can't improve it."**
>
> *"Jeśli nie możesz tego zmierzyć, nie możesz tego udoskonalić."*
> — Peter Drucker



## Dlaczego "Vibe Check" to za mało?

Zacznijmy od ustalenia faktów: *Vibe Check* (czyli manualne sprawdzenie, czy odpowiedź "wygląda dobrze") jest naturalnym pierwszym krokiem. Wszyscy tak zaczynamy. Kiedy budujesz proste "Hello World", patrzysz na wynik, widzisz sensowne zdanie i lecisz dalej.

Problem zaczyna się momentalnie, gdy wchodzimy w świat **Software 3.0**.

Twój system to już nie jest jeden prosty prompt. To skomplikowana maszyneria:
- Dynamiczne prompty systemowe (np. budowane w Jinja2)
- Wymuszone struktury wyjściowe (Structured Outputs/Pydantic)
- Baza wiedzy (RAG)
- Zewnętrzne narzędzia (Tools), do których model ma dostęp
- Pamięć, która zmienia się z każdą interakcją
- ...

### Efekt motyla w LLM
W tak złożonym środowisku pojawia się zjawisko, które nazywam [Efektem motyla](https://pl.wikipedia.org/wiki/Efekt_motyla). Wydaje Ci się, że zmieniasz tylko jedną, małą rzecz – na przykład dodajesz jedno pole do opisu w schemacie JSON albo poprawiasz literówkę w opisie narzędzia. Logika podpowiada: *"To wpłynie tylko na ten jeden element"*.

Błąd. Modele LLM nie działają liniowo.

Ta drobna zmiana w opisie pola może sprawić, że model nagle przestanie używać kalkulatora albo zacznie halucynować w zupełnie innej części dialogu. Zmienił się rozkład prawdopodobieństwa dla całego kontekstu.

Bez automatycznych przyrządów (ewaluacji), nie masz szans tego wyłapać.

### Problem skali i stabilności
Zastanówmy się nad przykładem  np. **AI Tutora** (nauczyciel języka).
Sprawdzenie manualne ("na oko") zadziała przy jednej wymianie zdań. Ale prawdziwa sesja nauki to dziesiątki, a nawet setki tur dialogu, wymiana wiedzy i kontekstu.

- **Skala:** Ile takich dialogów jesteś w stanie przeklikać po każdej zmianie w kodzie? Jeden? Dwa? Pięć? Przy piątym Twój mózg zaczyna ignorować szczegóły. A co z setnym przypadkiem brzegowym?
- **Powtarzalność:** Człowiek jest fatalnym testerem regresyjnym. Ty w poniedziałek rano (wypoczęty) i Ty w piątek wieczorem (zmęczony) ocenicie ten sam wynik inaczej. Jeśli nie masz powtarzalności pomiaru, nie masz inżynierii – masz zgadywanie.

Bez własnego leaderboarda i automatycznych testów (*golden dataset*), każda zmiana w kodzie to ryzyko. Nie wiesz, czy Twoja poprawka faktycznie coś naprawiła, czy może po cichu zepsuła 20% innych, kluczowych funkcji.

## Od demo do produkcji (rola metryk)

Ważna myśl: **bez twardych danych nie ma inżynierii.** Jest tylko hobbystyczne dłubanie.

Jeśli budujesz zabawny prototyp na weekendowy hackathon, "Vibe Check" wystarczy. Ale jeśli Twoim celem jest system *Production Ready* – taki, który ma działać stabilnie, skalować się i na siebie zarabiać – musisz wiedzieć, w którym miejscu jesteś.

Leaderboardy i benchmarki to Twoje przyrządy nawigacyjne. To one pozwalają Ci odpowiedzieć na pytanie: *"Czy zmiana modelu z GPT-5 na Claude 4 faktycznie poprawiła jakość, czy tylko zmieniła styl odpowiedzi?"*. To one decydują o tym, czy aktualizacja promptu systemowego to "optymalizacja", czy "sabotaż".

Zanim pójdziemy dalej, uporządkujmy słownik. Te pojęcia są kluczowe, bo w Software 3.0 budujemy systemy z klocków, a **Check** jest najmniejszym z nich:

1.  **Benchmark (Test):** To jest "egzamin". Cały zbiór zadań, pytań i oczekiwanych odpowiedzi (np. 1000 pytań z matematyki), który służy do sprawdzenia ogólnej umiejętności modelu.
2.  **Leaderboard (Ranking):** To jest "tablica wyników". Lista modeli posortowana od najlepszego do najgorszego na podstawie uśrednionych wyników z Benchmarku.
3.  **Check (punkt kontrolny / asercja):** To jest Twój "atom". Pojedynczy, konkretny test (jeden wiersz w Twoim zbiorze danych), który uruchamiasz na odpowiedzi modelu.
    *   *Przykład:* Budujesz AI Tutora. Twój Check to: *"Czy model zapytał użytkownika o imię?"*.
    *   Wynik Checku jest zazwyczaj binarny (Tak/Nie).
    *   W praktyce inżynierskiej grupujesz setki takich Checków w kategorie (np. "Gramatyka", "Empatia", "Bezpieczeństwo"), co pozwala Ci widzieć nie tylko ogólny wynik, ale precyzyjną mapę mocnych i słabych stron Twojego systemu.

W świecie AI mamy kilka "celebrytów" w kategorii publicznych *Benchmarków*:
- MMLU (*Massive Multitask Language Understanding*): Sprawdza ogólną wiedzę akademicką (od historii po chemię)
- HellaSwag: Teoretycznie bada "zdrowy rozsądek" (dokończ zdanie w logiczny sposób)
- SWE-bench: Ocenia zdolność do pisania kodu i rozwiązywania ticketów z GitHuba
- LMSYS Chatbot Arena: Ranking oparty na "ślepym" głosowaniu ludzi (Elo rating)
- i setki innych

### Wystarczy wziąć TOP 1?

Brzmi logicznie, prawda? Skoro mamy rankingi, to wystarczy wejść na Hugging Face Open LLM Leaderboard, posortować malejąco, wziąć model z samej góry i... *voila*! Mamy najlepsze rozwiązanie.

**Niestety, to tak nie działa.**

Gdyby to było takie proste, inżynierowie AI nie byliby potrzebni. Prawda jest taka, że publiczne leaderboardy, mimo że są krytycznie ważne dla rozwoju dziedziny, dla Ciebie – twórcy konkretnego produktu – są pełne szumu.

Dlaczego model, który "miażdży" konkurencję w benchmarku MMLU, może wyłożyć się na prostym zadaniu wyciągnięcia NIP-u z Twojej faktury? Dlaczego "najinteligentniejszy" model w rankingu SWE-bench może wprowadzić krytyczne błędy do Twojego repozytorium?

Aby to zrozumieć i nie dać się nabrać na marketingowy hype, musimy zajrzeć pod maskę tych rankingów. I to, co tam znajdziemy, może Cię mocno zaskoczyć.

## Dlaczego publiczne rankingi są podobne do kasyna?

Jako inżynierowie kochamy tabelki. Widzimy wykres, gdzie model X ma wynik 89.5, a model Y ma 87.2. Decyzja wydaje się prosta: bierzemy X, bo "cyferki nie kłamią". Niestety, w świecie LLM cyferki nie tylko kłamią – one aktywnie wprowadzają Cię w błąd.

Działa tu bezlitosne Prawo Goodharta:
> **"Kiedy miara staje się celem, przestaje być dobrą miarą."**

![](../images/goodhart.jpg)

W momencie, gdy pozycja #1 na [Chatbot Arena](https://lmarena.ai) zaczęła przekładać się na wyceny giełdowe i miliardowe inwestycje, leaderboard przestał być narzędziem naukowym. Stał się poligonem marketingowym.

### Jak Meta i Google hakują system?

LMSYS Chatbot Arena brzmi uczciwie: użytkownicy głosują na lepszą odpowiedź, wygrywa demokracja. Ale pod spodem dzieje się coś, co w inżynierii nazywamy *Selective Disclosure* (selektywnym ujawnianiem).

Badania ("[The Leaderboard Illusion](https://arxiv.org/pdf/2504.20879)") ujawniły brutalną prawdę o tym, jak giganci tacy jak Meta czy Google grają w tę grę. Wyobraź sobie, że idziesz na egzamin. Ale masz pewien przywilej: możesz podejść do niego 27 razy, w oddzielnych pokojach, a komisja zobaczy tylko Twój najlepszy wynik. Resztę palisz w piecu. Jesteś geniuszem? Nie, po prostu statystyka działa na Twoją korzyść.

To nie jest hipotetyczny przykład.

Badacze odkryli, że Meta testowała **27 prywatnych** wariantów modelu Llama przed wypuszczeniem:
1.  Wrzucają na arenę 27 modeli pod nazwami kodowymi (np. "iron-man", "thor", "hulk")
2.  Patrzą, który z nich najlepiej "smakuje" ludziom (nawet jeśli różnica jest błędem statystycznym)
3.  Publikują oficjalnie tylko zwycięzcę
4.  Reszta znika w niebyt (tzw. **score retraction**)

Dla Ciebie oznacza to jedno: Leaderboard nie mierzy jakości architektury modelu. Mierzy budżet firmy na trenowanie wariantów i optymalizację pod ranking.

![](../images/arent_privately_tested.jpg)


### Inteligentne lanie wody

Drugim grzechem głównym publicznych rankingów jest to, co lubią ludzie, a czego nienawidzą systemy produkcyjne.

Ludzie mają prostą heurystykę: "Dłuższa odpowiedź" = "Mądrzejsza odpowiedź"
Jeśli zapytasz o wynik `2+2`, a model A odpowie:
> "4"

A model B odpowie:
> "To fascynujące pytanie! W arytmetyce liczb naturalnych, dodawanie jest podstawową operacją. Biorąc pod uwagę aksjomaty Peano... wynik to z pewnością 4. Mam nadzieję, że to pomogło!"

...to statystyczny użytkownik Areny zagłosuje na model B. Wygląda "mądrzej", jest "miły".

W inżynierii Software 3.0 model B to katastrofa:
1.  **Koszt:** Płacisz za każdy wygenerowany token. Model B jest 10x droższy.
2.  **Latency:** Model B odpowiada 3 sekundy dłużej.
3.  **Parsowanie:** Wyciągnięcie cyfry "4" z tego bełkotu wymaga skomplikowanego wyrażenia regularnego albo kolejnego wywołania LLM.

Publiczne leaderboardy promują modele z "nadwagą" (bloated). Ty w swoim systemie potrzebujesz atlety, który robi zadanie i milknie. Dlatego model z miejsca #1 w rankingu niekoniecznie jest najlepszym wyborem dla Twojego systemu.

## Skażenie danych (*Data Contamination*)

Zauważyłeś ten cykl? Co tydzień nowy król.
Najpierw OpenAI pokazuje GPT-n. Chwilę później Anthropic wypuszcza Claude Y, który "miażdży" GPT-n na benchmarkach. Tydzień później wchodzi Google Gemini Z pro i "wygrywa" z Claude. Potem Grok X...


Czy naprawdę wierzymy, że co dwa tygodnie (miesiąca) dokonuje się fundamentalny przełom w dziedzinie sztucznej inteligencji? Czy te modele nagle stają się mądrzejsze?

Brutalna prawda inżynierska jest inna: **Wszyscy uderzyli w sufit.**

Obecna architektura (transformery) i dostępne metody treningu osiągnęły punkt nasycenia. Jeśli spojrzysz na "wielką trójkę" (OpenAI, Anthropic, Google) oraz ewentualnie Grok, to z perspektywy architektury stoją oni na dokładnie tej samej półce. Różnice, które widzisz na wykresach, to często szum statystyczny, łut szczęścia przy losowaniu nasiona (seed) lub mikro-optymalizacje.

Skoro więc technologia nie pozwala już na skokowy (znaczący) wzrost inteligencji, to w jaki sposób producenci "pompują" wyniki w tabelkach, żeby pokazać, że są lepsi od konkurencji? Odpowiedź jest prosta: Skoro nie możesz być mądrzejszy, musisz ... być sprytniejszy.

### Model widział arkusz przed egzaminem

Między innymi problemem jest sposób, w jaki trenujemy modele. Karmimy je "całym Internetem". A gdzie leżą najpopularniejsze zbiory testowe (benchmarki) takie jak [MMLU](https://huggingface.co/datasets/cais/mmlu), [GSM8K](https://huggingface.co/datasets/openai/gsm8k) czy [HumanEval](https://huggingface.co/datasets/openai/openai_humaneval)? 

Leżą na HuggingFace, GitHubie... w artykułach naukowych na ArXiv.

Dla modelu LLM to po prostu kolejny tekst do "przeczytania" podczas treningu. Kiedy więc prosisz model o rozwiązanie zadania z MMLU, on często nie wykonuje procesu myślowego (reasoning). On wykonuje proces przypominania sobie (*recall*).

To tak, jakby dać studentowi pytania egzaminacyjne wraz z odpowiedziami dzień przed sesją. To, że dostał 100%, nie oznacza, że rozumie fizykę kwantową. Raczej oznacza to, że ma świetną pamięć wzrokową.

### TS-Guessing (Test Set Slot Guessing)

Możesz powiedzieć: *"No dobra, ale może on jednak rozumie?"* To akurat można "wyłapać". Np. badacze z [Georgia Tech i Yale](https://arxiv.org/pdf/2311.09783) wymyślili genialny sposób, by to sprawdzić. Nazwali go **TS-Guessing**.

Wyobraź sobie zadanie wielokrotnego wyboru z benchmarku MMLU. Mamy pytanie, poprawną odpowiedź (np. D) i trzy błędne (A, B, C).
Badacze zrobili prosty trik: zakryli jedną z BŁĘDNYCH odpowiedzi i zapytali model: *"Co było tutaj wpisane?"*.

Z logicznego punktu widzenia, to zadanie niemożliwe. Błędna odpowiedź może być czymkolwiek.
Jeśli pytanie brzmi: *"Które miasto jest stolicą Francji?"*, a poprawne to *"Paryż"*, to błędną odpowiedzią może być *"Londyn"*, *"Radom"* albo *"Pizza hawajska"*. Nie da się tego wydedukować.

Chyba że... widziałeś ten arkusz wcześniej.

![](../images/ts-guessing.jpg)

### Porażające statystyki
Wyniki eksperymentu są miażdżące dla "inteligencji" modeli:

- **57% Exact Match:** GPT-4 potrafił odtworzyć **dokładną treść ukrytej, błędnej odpowiedzi** w 57% przypadków w teście MMLU.

#### Przykład z badania:
- Pytanie dotyczyło "aktorów niepaństwowych zagrażających USA". Poprawna odpowiedź to "Chiny". Jedną z błędnych opcji byli "Handlarze narkotyków".
- Badacze usunęli tekst "Handlarze narkotyków". Model zapytany o brakującą opcję C, wygenerował idealnie: *"Drug traffickers"*.

To jest dowód ostateczny. Model nie rozwiązywał zadania geopolitycznego. Model uzupełniał wzorzec tekstu (Canonical Order), który "przeczytał" w swoim zbiorze treningowym.

Przeczytaj to jeszcze raz, aby lepiej to zrozumieć.

#### Na czym polegał eksperyment?
Badacze nie sprawdzali, czy model potrafi odpowiedzieć na pytanie. Zrobili coś znacznie sprytniejszego. Sprawdzali, czy model **pamięta treść egzaminu**.

- **Zadanie:** Mamy pytanie wielokrotnego wyboru (A, B, C, D). Prawidłową odpowiedzią jest D (Chiny).
- **Test:** Badacze usunęli (zakryli) treść opcji C. Powiedzieli modelowi: „Oto pytanie, oto opcje A, B i D. Zgadnij, co było napisane w opcji C?”.
- **Opcja C to nie była poprawna odpowiedź!** Opcja C ("Drug traffickers" – handlarze narkotyków) była tzw. *dystraktorem* (zmyłką, błędną opcją).

#### Dlaczego wynik jest szokujący?
Model bezbłędnie uzupełnił opcję C jako "Drug traffickers".

Gdyby model używał **logiki i wiedzy o świecie**, to na prośbę "podaj przykład podmiotu niepaństwowego, który zagraża USA" (bo tym jest opcja C), mógłby wymyślić tysiące rzeczy:
- ISIS,
- Al-Kaida,
- Hakerzy,
- Kartele,
- Grupy paramilitarne,
- ...

Prawdopodobieństwo, że model "zgadnie", iż autorzy testu użyli akurat konkretnego sformułowania "Drug traffickers", jest statystycznie bliskie zeru. Chyba że model widział ten test wcześniej. No właśnie!

####  Dlaczego to jest "dowód ostateczny" 

Pomyślmy o tym jeszcze trochę... Model LLM (taki jak GPT) działa na zasadzie przewidywania kolejnego słowa. Uczy się na miliardach stron tekstu. Jeśli w jego zbiorze treningowym znalazły się zestawy pytań i odpowiedzi (np. z popularnych benchmarków do testowania AI), model potraktował je jak każdy inny tekst – np. jak wiersz czy piosenkę.

W tym przypadku mechanizm działania modelu wyglądał tak:
1.  Model widzi początek sekwencji (Pytanie + Opcja A + Opcja B)
2.  Model rozpoznaje ten wzorzec. To nie jest dla niego "zadanie z geopolityki". To jest dla niego "ten ciąg znaków, który występował w pliku `test_set_geopolitics.txt`"
3.  Model wie, że po Opcji B w tym konkretnym pliku następowały słowa "Drug traffickers"
4.  Wypisuje je

#### Wiedza vs pamięć
To jest dowód na to, że wysokie wyniki modeli w testach często nie wynikają tylko z ich "inteligencji", ale też z "pamięci fotograficznej".

To tak, jakbyś dał uczniowi do wypełnienia test z historii, w którym brakuje jednego słowa w pytaniu. Uczeń wpisuje słowo idealnie. Myślisz: "Ale on zna historię!". A w rzeczywistości uczeń po prostu nauczył się pytań z zeszłego roku na pamięć i recytuje je jak wierszyk, nie rozumiejąc nawet, o co pytają.

Dlatego model nauczył się, że w tym konkretnym tekście te konkretne słowa występują w tej konkretnej kolejności. Nie rozwiązał problemu geopolitycznego; dokończył wzorzec tekstu.

Zjawisko to nazywamy **Data Contamination** (skażeniem danych).

Wysokie wyniki w publicznych benchmarkach (szczególnie MMLU czy GSM8K) oznaczają dzisiaj głównie to, że inżynierowie z OpenAI czy Google nie odfiltrowali skutecznie zbiorów testowych ze swoich danych treningowych (albo zrobili to celowo słabo, żeby podbić wynik).

Dlatego w Software 3.0 przyjmujemy zasadę ograniczonego zaufania.

Nie namawiam Cię do całkowitego ignorowania publicznych metryk, zwłaszcza tych klasycznych jak MMLU. One wciąż pełnią funkcję ważnego "Sanity Check". Jeśli nowy model ma 30% skuteczności, od razu wiesz, że nie nadaje się do Twoich zastosowań. To świetny filtr wstępny, który oddziela modele "zepsute" od "używalnych".

Pułapka pojawia się na szczycie. W momencie, gdy czołowe modele dobijają do 90-95% (efekt saturacji), metryka traci swoją rozdzielczość. W tym rejonie nie jesteś już w stanie stwierdzić, czy wzrost o 1% to faktyczny skok inteligencji, czy po prostu mocniejsze skażenie danych treningowych. Traktuj więc leaderboardy (benchmarki) jako informację o **klasie modelu** (czy gra w lidze okręgowej, czy w lidze mistrzów), ale nigdy nie podejmuj decyzji o wdrożeniu tylko dlatego, że ktoś ma o 0.4 punktu procentowego więcej w tabelce.

## Absurd benchmarku HellaSwag: Kiedy "Lorem Ipsum" wystarczy

Żeby ostatecznie wyleczyć Cię z zaufania do cyferek w tabelkach, weźmiemy na warsztat **[HellaSwag](https://huggingface.co/datasets/Rowan/hellaswag)**. To jeden z popularnych benchmarków, który teoretycznie bada **Common Sense Reasoning** (zdrowy rozsądek). Tu jest pewna ironia, bo akurat zdrowego rozsądku zabrakło… ale po kolei wyjaśnię, o co chodzi. Jak to często bywa, wizja (idea) była całkiem ciekawa, ale coś poszło nie tak.

Mamy sytuację (np. *"Kobieta na siłowni podnosi ciężary..."*) i model musi wybrać logiczne zakończenie zdania spośród czterech opcji.

W jaki [sposób zweryfikowano](https://arxiv.org/pdf/2504.07825), czy model faktycznie "rozumuje", czy tylko cwaniakuje. Przeprowadzili eksperyment, który wstrząsnął fundamentami ewaluacji.

### Eksperyment "Zero-Prompt" 

Procedura wyglądała tak:
1.  **Standard:** Model dostaje Pytanie (kontekst) + 4 Odpowiedzi. Wybiera poprawną
2.  **Zero-Prompt:** Usuwamy Pytanie całkowicie. Model widzi **tylko 4 Odpowiedzi**
3.  **Lorem Ipsum:** Zamiast pytania wstawiamy losowy tekst "Lorem ipsum dolor...".

Wynik?
Okazało się, że w **ok. 68% przypadków** model wybiera dokładnie tę samą odpowiedź, niezależnie od tego, czy widzi pytanie, czy nie. W wielu przypadkach usunięcie pytania wręcz **poprawia** wynik.

### 🤯 Zatrzymaj się i pomyśl o tym
Jak można poprawnie dokończyć zdanie, którego początku nie znasz?
To tak, jakbym kazał Ci wybrać "poprawne zakończenie historii", nie opowiadając historii, a Ty mimo to trafiasz w 80%. Magia? Nie. Inżynieria wsteczna.

### Dlaczego to działa? 

`HellaSwag` został stworzony lata temu przy użyciu metody **Adversarial Filtering** z wykorzystaniem starych modeli (era GPT-1/GPT-2). Przypomnę, że pierwszym „ChatGPT” był GPT-3.5 (od tego momentu zaczął się boom). Wersja GPT-3, a tym bardziej GPT-2, była znacznie gorsza.
- **Poprawna odpowiedź** pochodzi z prawdziwego tekstu (ActivityNet/WikiHow) – jest naturalna, gramatyczna i płynna.
- **Błędne odpowiedzi** (distractors) zostały wygenerowane przez słaby model AI, żeby zmylić inny model.

Efekt? Błędne odpowiedzi są pełne "artefaktów": literówek, błędów gramatycznych (np. *"people is standing"* zamiast *"people are standing"*) i nienaturalnych konstrukcji.

Dzisiejsze modele są o rzędy wielkości potężniejsze niż modele, które generowały ten test. Dla nich to zadanie nie polega na "rozumieniu kontekstu siłowni". Polega na prostym **klasyfikatorze stylu**:
- Opcja A: Bełkot syntetyczny (odrzuć)
- Opcja B: Bełkot syntetyczny (odrzuć)
- Opcja C: Naturalny język angielski (wybierz)
- Opcja D: Bełkot syntetyczny (odrzuć)

Model nie musi wiedzieć, o co pytasz. Wystarczy, że widzi, która odpowiedź jest napisana poprawną polszczyzną (czy angielszczyzną).

Dlatego  `HellaSwag` na leaderboardzie nie mierzy "Common Sense Reasoning". On mierzy zdolność modelu do wykrywania tekstu generowanego przez stare modele AI.

To klasyczny przykład *Construct Validity Issue*. Benchmark mierzy coś zupełnie innego, niż głosi jego nazwa. Jeśli optymalizujesz swój model pod HellaSwag, nie uczysz go myślenia – uczysz go bycia korektorem gramatycznym dla tekstów z 2019 roku. Szansa, że trafisz na takie zadanie, jest praktycznie zerowa.

## Mit "AI Programisty" 

Pamiętasz szał na **Devina**? "Pierwszy AI Software Engineer". Internet obiegły wykresy, na których modele osiągały 50%, a potem nawet 70% skuteczności w rozwiązywaniu zadań z GitHuba.
Szefowie firm zacierają ręce – "zwolnimy juniorów".

Jako inżynier musisz wylać im na głowy kubeł zimnej wody. Te wyniki są **zawyżone** z powodów czysto technicznych. SWE-bench, choć potrzebny, stał się ofiarą Prawa Goodharta. 


![](../images/devin_swe.jpg)

O co chodzi?


### Absurd 1: Test "Ślepego Strzelca"

[Badanie](https://arxiv.org/pdf/2512.10218) przeprowadziło prosty eksperyment. Zadali modelom (Claude 3.5/3.7) pytanie: *"W którym pliku znajduje się błąd?"*, dając im tylko treść zgłoszenia (issue).
Modele **nie dostały** listy plików ani struktury repozytorium.

Logicznie rzecz biorąc, to zadanie niemożliwe. To tak, jakbym powiedział: *"Mam błąd w formularzu logowania"* i kazał Ci podać ścieżkę do pliku na dysku mojego komputera, którego nie widzisz.

Wyniki?
- Na zbiorze **SWE-bench Verified** model Claude 3.5 wskazał poprawnie wszystkie pliki w aż **65%** przypadków.
- Na zbiorze kontrolnym (**BeetleBox** - podobne repozytoria Pythonowe, ale spoza benchmarku), skuteczność spadła do **12%**

[![](../images/swe_bench_verified.jpg)](https://arxiv.org/pdf/2512.10218)


Co to oznacza? 

Model nie "analizuje problemu". On po prostu pamięta, że w *scikit-learn* błąd z "RidgeClassifier" naprawiało się w pliku `linesearch.py`. To nie jest inteligencja, to *data leakage*.



### Absurd 2: Karaoke kodowania (31.6% Verbatim Match)

Inne badanie ([The SWE-Bench Illusion: When State-of-the-Art LLMs Remember Instead of Reason](https://arxiv.org/pdf/2506.12286)) poszło o krok dalej. Sprawdzili, w jakim stopniu modele generują kod "z głowy" (rozumowanie), a w jakim odtwarzają ciągi znaków zapamiętane z treningu.

Użyli metryki **5-gram overlap** (pokrycie ciągów 5 tokenów).
- Dla zadań z SWE-bench Verified pokrycie z oryginałem wynosiło aż **35%**
- Dla zadań spoza tego zbioru (ale z tych samych repozytoriów!) spadało do **18%**

[![](../images/swe_verbatium.jpg)](https://arxiv.org/pdf/2506.12286)

Co gorsza, model **Claude 4 Opus w 31.6% przypadków** odtwarzał kod rozwiązania **verbatim** (słowo w słowo), mając podany tylko prefiks pliku. To nie jest programowanie. To jest karaoke. Model "śpiewa" kod, który zna na pamięć.



Natomiast zobacz, jak to fajnie brzmi przy kolejnym release: mamy najlepszy wynik w SWE!
Teraz nie chodzi o to, aby krytykować Anthropic (albo przynajmniej tylko ich). Bardziej pokazuję Ci, jaka gra się tutaj odbywa i że dostawcom zależy na pokazaniu postępu. I na tym zależy im bardziej niż na dostarczeniu Ci LLM-a, który rzeczywiście działa dobrze. I znów warto przypomnieć cytat, który już dzisiaj się pojawił:

> **"Kiedy miara staje się celem, przestaje być dobrą miarą."**

![](../images/claude_swe_benchmark.jpg)


#### A co z wersją "SWE-bench Verified"?

Możesz zapytać: *"Ale zaraz, przecież OpenAI wypuściło [wersję Verified](https://openai.com/index/introducing-swe-bench-verified/), która naprawia te błędy!"*.

To prawda i nieprawda.

OpenAI wykonało tytaniczną pracę, zatrudniając 93 programistów do ręcznego sprawdzenia zadań. Wynik? **Odrzucili aż 68.3% oryginalnych zadań** jako wadliwe (zbyt niejasne opisy, błędne testy, problemy z konfiguracją). To pokazuje, jak słabej jakości był oryginalny benchmark.

Wersja **Verified** (500 zadań) jest więc technicznie poprawna – zadania są wykonalne, a testy uczciwe. Dlatego wyniki modeli skoczyły tam dwukrotnie (np. GPT-4o z 16% na 33.2%).


### 👆 Zwróć uwagę
Wersja *Verified* naprawia problem **jakości testów** (Validity), ale nie naprawia problemu **skażenia danych** (Contamination). Wszystkie miażdżące statystyki, które przytoczyłem wyżej (spadek skuteczności na nieznanych repozytoriach), zostały przeprowadzone właśnie na *SWE-bench Verified*. Model teraz nie wywala się na błędnych testach, ale nadal rozwiązuje te zadania głównie z pamięci. "Verified" to lepszy egzamin, ale student nadal zna pytania.


### Absurd 3: "Oracle Mode" i wycieki rozwiązania

Wiele imponujących wyników na leaderboardach jest osiąganych w trybie **"Oracle"** (Golden Retrieval).
W realnym życiu 80% Twojego czasu to znalezienie błędu. W trybie Oracle model dostaje na tacy informację: *"Edytuj plik `models.py`"*. Bez tej podpowiedzi (tryb *Unassisted*), skuteczność spada drastycznie.

Dodatkowo, analizy (np. *SWE-Bench+*) wykazały, że w 1/3 przypadków "sukcesu", gotowe rozwiązanie (lub dokładny kod patcha) znajdowało się wprost w treści *Issue* lub komentarzach, które model dostał na wejściu. Model działał wtedy jak kserokopiarka, a nie inżynier.


Dlatego, znów, nie ufaj wynikom typu "50% zadań rozwiązanych".

Realna zdolność modelu do naprawienia błędu w Twoim prywatnym, zamkniętym repozytorium (którego model nie widział w treningu) jest ułamkiem tego, co widzisz na SWE-bench. Traktuj ten benchmark jako test pamięci i obsługi gita, a nie test inżynierii.

## Co z tym zrobić? 

Skoro już wiesz, że publiczne rankingi są jak mapa zniekształcona przez marketing, pojawia się pytanie: **Jak żyć?** Czy mamy ignorować premiery nowych modeli? Czy mamy zakładać, że każdy nowy "SOTA" to kłamstwo?

Absolutnie nie.

Publiczne benchmarki to wciąż ważny sygnał. Jeśli model nie potrafi rozwiązać prostego MMLU, to prawdopodobnie nie nada się do niczego. Ale jeśli ma 99%? Tu zaczynają się schody.

### Lekcja z "Llama 4 Gate"

Spójrzmy na konkretną sytuację, która idealnie obrazuje ten problem. Wyobraź sobie, że wychodzi `Llama 4`. Wykresy strzelają w górę, słupki przebijają sufit. PR Meta ogłasza zwycięstwo. A chwilę później na Reddicie pojawia się [taki wpis](https://www.reddit.com/r/LocalLLaMA/comments/1jt8yug/serious_issues_in_llama_4_training_i_have/) (rzekomo od pracownika):

> **"Poważne problemy z treningiem Llama 4. Złożyłem rezygnację."**
> (...) Kierownictwo sugerowało mieszanie zbiorów testowych z różnych benchmarków podczas procesu post-treningu, aby osiągnąć cele w metrykach i uzyskać "reprezentatywny" wynik. (...) Użytkownicy na X i Reddicie zgłaszają fatalne wyniki w rzeczywistym użyciu.

Co robisz z taką informacją?

### Opcja 1: Droga komentatora (Nie polecam)
Wchodzisz na Twittera/X i Reddita. Czytasz setki komentarzy.
- Użytkownik A: "U mnie działa super!"
- Użytkownik B: "Totalne dno, halucynuje."
- Użytkownik C: "To spisek!"

Marnujesz 4-10-100 godzin na czytanie anegdotycznych dowodów ("Vibe Check" innych ludzi). Na koniec dnia nadal nie wiesz, czy możesz użyć tego modelu w produkcji.

### Opcja 2: Droga w Software 3.0 (jedyna słuszna)
Ignorujesz szum. Masz to gdzieś, co piszą na Reddicie (traktujesz jako ciekawostka)

Masz swój **własny Leaderboard** (Golden Dataset).
1.  Pobierasz nowy model (lub podpinasz API).
2.  Uruchamiasz swój zestaw 50 testów specyficznych dla **Twojego** problemu.
3.  Po 5 minutach masz wynik: *"W moim zadaniu wyciągania danych z faktur, Llama 4 ma skuteczność 82%, podczas gdy stary model miał 78%. Jest lepiej, ale gorzej z formatowaniem JSON."*

Zobacz, co zrobiliśmy w DataWorkshop Lab tego dnia, gdy pojawiła się LLaMA 4. Sprawdziliśmy, jak wypada na naszym leaderboardzie (dla konkretnego problemu).

![](../images/llama4_leaderboard.jpg)

Na ten moment, kiedy wyszedł model, wynik był bardzo dobry jak na otwarte modele (wyżej był tylko `DeepSeek R1`). Co więcej, nastąpiła znacząca poprawa wsparcia Structured Output. Nawet dla LLaMA 3 (tej największej) wsparcie to nie było takie łatwe — często trzeba było wykonywać wiele prób, aby w końcu otrzymać poprawną odpowiedź. Między innymi w naszym teście model wypadł nawet nieco lepiej niż `gpt-4o`. Różnica była minimalna, ale jednak była.

Koniec dyskusji. Masz dane. Podejmujesz decyzję.

## Dlaczego Twój Leaderboard > Publiczny Leaderboard?

Budując własny zestaw testów, zyskujesz supermoc, której nie da Ci żadna Chatbot Arena: **zaufanie**

Publiczny ranking mówi: *"Model A jest lepszy od B"*.

Twój ranking mówi: *"Model A świetnie radzi sobie z kreatywnym pisaniem, ale myli się w obliczeniach. Model B jest sztywny, ale bezbłędnie generuje SQL."*

Mało tego, jak najbardziej może się okazać, że "najlepszy model świata" jest **gorszy** w Twoim specyficznym wąskim gardle niż mniejszy, tańszy model, który akurat "czuje" Twoją domenę.

### Zarządzanie złożonością

W Software 3.0 nieustannie kręcisz gałkami:
- Zmieniasz prompt systemowy
- Dodajesz warunki dynamiczne w promcie
- Zmienia strukturę (Structured Output)
- Zmieniasz warstwę RAG
- Zmieniasz warstwę pamięci czy narzędzia
- Dodajesz adaptowalne plany
- Zmieniasz model 
- ...

Każda z tych zmian to ruch w wielowymiarowej przestrzeni. Bez własnego leaderboarda (zestawu binarnych testów [*check*]) błądzisz po omacku. Może poprawisz styl odpowiedzi, ale zepsujesz formatowanie daty? Może model jest teraz "mądrzejszy", ale przestał odpowiadać na pytania w języku polskim?

Mając własny **Golden Dataset**, zmiana modelu to nie "ryzykowna operacja na otwartym sercu", ale rutynowa procedura inżynierska. Puszczasz testy. Wszystko zielone? Wdrażasz. Coś czerwone? Rollback.

To właśnie daje Ci kontrolę i sprawia, że w rozmowie z biznesem lub zespołem nie używasz zwrotów *"wydaje mi się"*, tylko *"dane pokazują"*.

## Sędzia, który nie śpi (LLM-as-a-Judge)

Doszliśmy do momentu, w którym masz już swój *Golden Dataset* (zbiór pytań). Teraz musisz odpowiedzieć na pytanie: **Jak sprawdzić, czy odpowiedź jest poprawna?**

W klasycznym programowaniu (Software 1.0) sprawa jest prosta. Masz asercję:
```python
assert result == "expected_string"
```
W Software 3.0 to nie zadziała. Model jest niedeterministyczny.
Jeśli oczekujesz odpowiedzi *"Tak"*, a model odpowie *"Oczywiście, że tak"* albo *"Potwierdzam"* – klasyczny `assert` rzuci błędem, mimo że logika jest poprawna.

Potrzebujesz mechanizmu, który rozumie **semantykę**, a nie tylko składnię.
Potrzebujesz **LLM-as-a-Judge**.


Technicznie rzecz biorąc, to po prostu kolejne wywołanie API. Wysyłasz do modelu odpowiedź, którą chcesz ocenić, wraz z instrukcją oceniania. To, co brzmi prosto, wcale nie jest takie łatwe w realizacji.

```

╔═══════════════════════════════════════════════════════════════════════════╗
║                          LLM AS JUDGE CONCEPT                             ║
╚═══════════════════════════════════════════════════════════════════════════╝

                              EVALUATION FLOW
                              ===============

    ┌─────────────┐         ┌─────────────────┐         ┌─────────────┐
    │   INPUT     │         │  MODEL UNDER    │         │   OUTPUT    │
    │   PROMPT    │────────▶│   EVALUATION    │────────▶│  RESPONSE   │
    │             │         │   (Candidate)   │         │             │
    └─────────────┘         └─────────────────┘         └──────┬──────┘
                                                               │
                                                               │
                    ┌──────────────────────────────────────────┘
                    │
                    ▼
    ╔═══════════════════════════════════════════════════════════════════╗
    ║                                                                   ║
    ║                        🏛️  LLM JUDGE  🏛️                          ║
    ║                                                                   ║
    ║  ┌─────────────────────────────────────────────────────────────┐  ║
    ║  │                      INPUTS RECEIVED:                       │  ║
    ║  │                                                             │  ║
    ║  │   • Original Prompt/Question                                │  ║
    ║  │   • Generated Response (to evaluate)                        │  ║
    ║  │   • Evaluation Criteria / Rubric                            │  ║
    ║  │   • [Optional] Reference Answer (ground truth)              │  ║
    ║  └─────────────────────────────────────────────────────────────┘  ║
    ║                              │                                    ║
    ║                              ▼                                    ║
    ║  ┌─────────────────────────────────────────────────────────────┐  ║
    ║  │                   EVALUATION PROCESS:                       │  ║
    ║  │                                                             │  ║
    ║  │   1. Analyze response against criteria                      │  ║
    ║  │   2. Check for accuracy, relevance, coherence               │  ║
    ║  │   3. Identify strengths and weaknesses                      │  ║
    ║  │   4. Generate structured assessment                         │  ║
    ║  └─────────────────────────────────────────────────────────────┘  ║
    ║                                                                   ║
    ╚═══════════════════════════════════════════════════════════════════╝
                                   │
                                   ▼
    ┌───────────────────────────────────────────────────────────────────┐
    │                        JUDGE OUTPUT                               │
    │  ┌─────────────────┐  ┌─────────────────┐  ┌──────────────────┐   │
    │  │     SCORE       │  │   EXPLANATION   │  │  RECOMMENDATIONS │   │
    │  │   (Pass/Fail)   │  │   (Reasoning    │  │  (How to improve │   │
    │  │                 │  │    behind score)│  │   the response)  │   │
    │  └─────────────────┘  └─────────────────┘  └──────────────────┘   │
    └───────────────────────────────────────────────────────────────────┘


═══════════════════════════════════════════════════════════════════════════
                          COMMON USE CASES
═══════════════════════════════════════════════════════════════════════════

┌─────────────────────────────────────────────────────────────────────────┐
│                                                                         │
│  1. POINTWISE EVALUATION          2. PAIRWISE COMPARISON                │
│     (Single Response)                (Two Responses)                    │
│                                                                         │
│     ┌──────────┐                     ┌──────────┐   ┌──────────┐        │
│     │Response A│                     │Response A│   │Response B│        │
│     └────┬─────┘                     └────┬─────┘   └────┬─────┘        │
│          │                                │              │              │
│          ▼                                └──────┬───────┘              │
│     ┌──────────┐                                 ▼                      │
│     │  JUDGE   │                          ┌──────────┐                  │
│     └────┬─────┘                          │  JUDGE   │                  │
│          │                                └────┬─────┘                  │
│          ▼                                     │                        │
│     Score: True                                ▼                        │
│                                          "A is_better"                  │
│                                          or "B wins"                    │
│                                                                         │
└─────────────────────────────────────────────────────────────────────────┘

```


### Koniec z "Oceną 1-5"

Wielu inżynierów instynktownie sięga po skale numeryczne.
*"Oceń w skali 1-10, jak pomocna jest ta odpowiedź"*.

Zwykle to pułapka. 

Brutalna prawda jest taka, że skale wprowadzają "odcienie szarości", które zabijają powtarzalność testów.
Dlaczego?
1.  **Brak kalibracji:** Czym dla modelu (i dla Ciebie) różni się ocena 7 od 8? Granica jest płynna. To, co rano wydaje się "mocną siódemką", wieczorem może być "słabą ósemką".
2.  **Szum stochastyczny:** Jeśli dajesz modelowi elastyczność (10 stopni swobody), zwiększasz szansę, że wpadnie w losową koleinę. Uruchomisz test 3 razy i dostaniesz: 7, 8, 6. Czy Twój system działa lepiej? Nie wiesz.
3.  **Brak decyzyjności:** W procesie CI/CD (Continuous Integration) test musi przejść (**Pass**) albo nie (**Fail**). Co zrobisz z wynikiem 3.5? Przepuścisz go na produkcję?

### Filozofia binarnych warunków (*Binary Checks*)

W inżynierii systemów agentowych dążymy do **binarności**.
Chcemy, aby Sędzia miał tylko dwie opcje: **TAK (1)** lub **NIE (0)**. To drastycznie redukuje szum i wymusza jednoznaczność.

To prosta, ale genialna idea, która broni się w boju. Jednak wdrożenie jej wymaga zmiany myślenia.
Sztuka polega na tym, aby Twoje subiektywne oczekiwanie ("Model ma być miły") rozbić na zestaw twardych, binarnych pytań.

Zobacz różnicę:

| Złe podejście (Skala / Vibe) | Dobre podejście (Binarne Czeki) |
| :--- | :--- |
| "Oceń ton wypowiedzi (1-5)" | "Czy model użył zwrotów grzecznościowych?" (T/N) |
| "Sprawdź poprawność danych" | "Czy kwota na fakturze zgadza się z kwotą w bazie?" (T/N) |
| "Czy odpowiedź jest bezpieczna?" | "Czy odpowiedź zawiera zachętę do przemocy?" (T/N) |
| "Czy format jest OK?" | "Czy odpowiedź jest poprawnym JSON-em?" (T/N) |

Jeśli rozbijesz swój problem na 50 takich pytań, nagle okazuje się, że masz precyzyjną mapę jakości. Jeśli model zaliczył 48 na 50 czeków, masz wynik 96%. To jest twarda liczba, której możesz zaufać, a nie "subiektywne 9/10".

Swoją drogą, LLM świetnie pomaga w tej konwersji – możesz poprosić go: *"Mam takie ogólne wymaganie X, rozbij je na 5 binarnych kryteriów testowych"* – ale ostateczna weryfikacja należy do Ciebie.


Pamiętaj, że w Software 3.0 nie zgadujemy, co model nam zwróci – my to definiujemy (poziom 2 i wyżej). Zanim zatrudnimy naszego cyfrowego sędziego, musimy stworzyć dla niego strukturę, który będzie wypełniał. Jeśli zrobisz to dobrze na poziomie Pydantic, zaoszczędzisz godziny na debugowaniu promptów. Bierzmy się do pracy.

## Wczytujemy biblioteki

In [1]:
from pydantic import BaseModel, Field
import helper as h

## 🧠 Zadanie 5.1.1: Struktura werdyktu

Zanim zatrudnimy sędziego, musimy zaprojektować formularz, który będzie wypełniał. W świecie Software 3.0 nie parsujemy tekstu przy pomocy RegExów. Definiujemy schemat danych, którego model musi przestrzegać.

Kluczową techniką, którą tu zastosujemy, jest **Critique Shadowing**.
Jeśli zapytasz model tylko o wynik (Tak/Nie), strzeli on odpowiedzią "z biodra" (bez zastanowienia). Jeśli wymusisz na nim najpierw napisanie uzasadnienia, a potem werdyktu – jakość oceny drastycznie wzrośnie. Kolejność pól w klasie ma znaczenie!

### Twoim zadaniem jest:
Zdefiniowanie klasy `Judgement` przy użyciu biblioteki `Pydantic`.
Klasa musi posiadać dwa pola:
1.  `reasoning`: Miejsce na proces myślowy sędziego (analizę).
2.  `is_correct`: Binarny werdykt.

**Wymaganie krytyczne:** Użyj `Field(description="...")` dla każdego pola. To nie są komentarze dla programisty – to jest część promptu dla modelu! Opisz dokładnie, co model ma tam wpisać.

<details>
    <summary style="background: #e6eaeb; padding: 4px 0; text-align: center; font-size: 20px; font-weight: 900;">
    👉 Kliknij tutaj, aby zobaczyć podpowiedź
    </summary>
<p>

### Wskazówki:

1.  Użyj `BaseModel` i `Field` z `pydantic`.
2.  Pamiętaj, że LLM generuje tokeny sekwencyjnie. Jeśli pole `is_correct` dasz jako pierwsze, model podejmie decyzję **zanim** ją przeanalizuje. To błąd. Pole `reasoning` musi być pierwsze.
3.  W opisie pola `is_correct` zaznacz, że interesuje nas twarda binarność (True/False).

<details>
    <summary style="background: #e6eaeb; padding: 4px 0; text-align: center; font-size: 20px; font-weight: 900;">
    👉 Kliknij tutaj, aby zobaczyć rozwiązanie
    </summary>
<p>

```python

class Judgement(BaseModel):
    # KROK 1: Najpierw myślenie (Chain of Thought)
    reasoning: str = Field(
        description="Krótkie, logiczne uzasadnienie werdyktu. Przeanalizuj odpowiedź krok po kroku pod kątem kryteriów. Wskaż ewentualne błędy."
    )
    
    # KROK 2: Potem decyzja (Binary Verdict)
    is_correct: bool = Field(
        description="Ostateczny werdykt. True, jeśli odpowiedź spełnia WSZYSTKIE kryteria (Pass). False, jeśli występuje jakikolwiek błąd (Fail)."
    )
```
</p>
</details>
</p>
</details>

In [2]:
## YOUR CODE HERE
class Judgement(BaseModel):
    reasoning: str = Field(
        ...,
        description="PLace for inner reasoning. You need to provide justification before giving answer"
    )
    is_correct: bool = Field(
        ...,
        description = "Final answer. TRUE if answer pass all critieria (PASS). False if there is any mistake (Fail)"
    )

Dobra, mamy już strukturę danych, która wymusza na modelu odpowiedni format i kolejność myślenia. To jest nasz fundament. Teraz możemy przejść dalej i wpiąć tę strukturę do konkretnego wywołania API. Musimy napisać funkcję, która nada modelowi odpowiedni kontekst (System Prompt) i zmusi go do bycia surowym recenzentem, a nie "pomocnym asystentem".

## 🧠 Zadanie 5.1.2: Pętla sędziego 

Mamy już "druk formularza" (klasę `Judgement`). Teraz musimy napisać instrukcję dla urzędnika (Sędziego), jak ma go wypełnić, i wysłać to do "centrali" (API).

To zadanie polega na inżynierii promptu. Sędzia musi być bezlitosny. Jeśli model będzie "miły" i zaliczy odpowiedź, która jest "prawie dobra", cały nasz system ewaluacji staje się bezużyteczny.

### Twoim zadaniem jest:
Napisanie funkcji `evaluate_response`, która:
1.  Konstruuje **System Prompt** z użyciem tagów XML (np. `<rules>`), narzucając rolę surowego sędziego.
2.  Konstruuje **User Message**, który czytelnie oddziela zapytanie, odpowiedź modelu i kryteria oceny (również używając tagów XML, np. `<model_output>`).
3.  Wykonuje strzał do API używając `h.call_api` ze strukturą z poprzedniego zadania.

<details>
    <summary style="background: #e6eaeb; padding: 4px 0; text-align: center; font-size: 20px; font-weight: 900;">
    👉 Kliknij tutaj, aby zobaczyć podpowiedź
    </summary>
<p>

### Algorytm:

1.  W **System Prompt** zawrzyj zasady:
    *   Bądź obiektywny.
    *   Najpierw `reasoning`, potem `is_correct`.
    *   Ignoruj "laniem wody" (fluff).
2.  W **User Message** sklej stringi `user_query`, `model_output`, `criteria`. Użycie tagów XML (np. `<input>... </input>`) pomaga modelowi zrozumieć, gdzie kończy się pytanie, a zaczyna odpowiedź do oceny (zapobiega to tzw. *Prompt Injection* z treści ocenianej).
3.  Użyj `h.call_api(messages, response_model=Judgement)`.

<details>
    <summary style="background: #e6eaeb; padding: 4px 0; text-align: center; font-size: 20px; font-weight: 900;">
    👉 Kliknij tutaj, aby zobaczyć rozwiązanie
    </summary>
<p>

```python
def evaluate_response(user_query: str, model_output: str, criteria: str) -> Judgement:
    
    # 1. System Prompt: Definicja roli i zasad
    system_prompt = """
    Jesteś bezstronnym i surowym sędzią AI (LLM-as-a-Judge). 
    Twoim zadaniem jest weryfikacja, czy odpowiedź modelu spełnia narzucone kryteria binarne.
    
    <rules>
        1. Bądź krytyczny. Nie przepuszczaj odpowiedzi "prawie dobrych".
        2. Zastosuj 'Critique Shadowing': Najpierw wypełnij pole 'reasoning', a dopiero potem wydaj werdykt w 'is_correct'.
        3. Ignoruj uprzejmości ("Sure, I can help"). Liczą się twarde fakty.
        4. Jeśli kryterium nie jest spełnione w 100%, werdykt to False.
    </rules>
    """
    
    # 2. User Message: Separacja kontekstu
    user_message = f"""
    Oceń poniższą interakcję:

    <user_query>
    {user_query}
    </user_query>
    
    <model_output>
    {model_output}
    </model_output>
    
    <criteria>
    {criteria}
    </criteria>
    """

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_message}
    ]

    # 3. Wywołanie API ze Structured Output
    # Zakładamy, że klasa Judgement jest dostępna z poprzedniego zadania
    return h.call_api(messages=messages, response_model=Judgement)
    
```
</details>
</p>
</details>

In [3]:
## YOUR CODE HERE
def evaluate_response(user_query: str, model_output: str, criteria: str) -> Judgement:

    system_prompt = """
        Jesteś bezstronnym i surowym sędzią AI (LLM-as-a-Judge). 
    Twoim zadaniem jest weryfikacja, czy odpowiedź modelu spełnia narzucone kryteria binarne.
    
    <rules>
        1. Bądź krytyczny. Nie przepuszczaj odpowiedzi "prawie dobrych".
        2. Zastosuj 'Critique Shadowing': Najpierw wypełnij pole 'reasoning', a dopiero potem wydaj werdykt w 'is_correct'.
        3. Ignoruj uprzejmości ("Sure, I can help"). Liczą się twarde fakty.
        4. Jeśli kryterium nie jest spełnione w 100%, werdykt to False.
    </rules>
    
    """
    # 2. User Message: Separacja kontekstu
    user_message = f"""
    Oceń poniższą interakcję:

    <user_query>
    {user_query}
    </user_query>
    
    <model_output>
    {model_output}
    </model_output>
    
    <criteria>
    {criteria}
    </criteria>
    """

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_message}
    ]

    # 3. Wywołanie API ze Structured Output
    # Zakładamy, że klasa Judgement jest dostępna z poprzedniego zadania
    return h.call_api(messages=messages, response_model=Judgement)

## Uruchomiamy

Mamy już komplet: zdefiniowaną strukturę danych oraz gotową funkcję `evaluate_response`. To teraz czas na "chwilę prawdy". Możemy to wywołać.

Przygotowałem dla Ciebie konkretny scenariusz testowy: zapytanie o finanse i odpowiedź modelu, który ewidentnie "leje wodę", unikając konkretów. Zdefiniowałem też twarde kryterium. Zobaczmy, czy nasz sędzia to wyłapie.

In [4]:
q = "Jaki jest kapitał zakładowy spółki X?"
resp = "Spółka X to lider rynku..." # Lanie wody
crit = "Musi być podana kwota."

verdict = evaluate_response(q, resp, crit)
print(f"Werdykt: {verdict.is_correct}") 
print(f"Powód: {verdict.reasoning}")

Werdykt: False
Powód: Model output does not provide the specific information requested by the user, which is the amount of the share capital of Company X. Instead, it provides general information about Company X, which does not satisfy the requirement of stating the share capital amount. Since the criterion specifies that a numeric amount must be given, the response is incomplete.


No i tak jak widzisz, sędzia powiedział, że jest `FAIL` (`False`). Co więcej, w polu reasoning precyzyjnie wytłumaczył, dlaczego tak uznał – wskazując na brak konkretnej kwoty i ogólnikowość wypowiedzi. To jest właśnie ta jakość, której szukamy: automatyczna, ale z uzasadnieniem.



### 👆 Zwróć uwagę
Jak precyzyjnie zadziałał ten mechanizm. Nie dostaliśmy rozlazłej opinii, tylko twardy `boolean`: **False**.

Wyobraź sobie alternatywę – poziom 1, czyli zwykła rozmowa z modelem. Pytasz: *"Czy ta odpowiedź jest poprawna?"*.
Model odpisuje: *"Cóż, odpowiedź jest uprzejma i gramatycznie poprawna, ale w sumie nie zawiera konkretnej kwoty, chociaż z drugiej strony..."*.

I co z tym zrobisz w kodzie?
Będziesz pisał `if "w sumie" in response`? To jest droga donikąd.

Gdy uruchamiasz jeden test, możesz sobie pozwolić na czytanie esejów. Ale gdy budujesz system produkcyjny, będziesz uruchamiał tych testów **tysiące**. Potrzebujesz dashboardu, w którym jest jednoznaczność, który świeci na zielono (True) albo na czerwono (False).

Tylko wtedy wynik staje się zarządzalny, a na tym nam bardzo zależy, prawda? Tylko wtedy możesz powiedzieć: *"Mój system działa poprawnie na 87%"*, a nie *"Mój system działa chyba nieźle, bo model tak napisał"*.

## Podsumowanie

Podsumujmy to, co dzisiaj odkryliśmy, bo to fundament pod wszystko, co będziemy budować dalej.

Brutalna prawda jest taka, że jeśli chcesz tworzyć produkty oparte na LLM, a nie tylko efektowne demo, musisz przestać ufać publicznym rankingom. Zrozumieliśmy, że leaderboardy są pełne szumu marketingowego i zjawiska skażenia danych, gdzie modele "recytują" odpowiedzi z pamięci, zamiast faktycznie rozumować. Dlatego traktowanie ich jako wyroczni przy wyborze modelu to inżynierskie samobójstwo.

Rozwiązaniem jest budowa własnego leaderboarda opartego o Twój unikalny kontekst. To konieczność, choć — jak zauważyłeś — wcale nie jest to trywialne zadanie. Technicznie sprowadza się to do wykorzystania LLM-a jako sędziego (LLM-as-a-Judge). Zrobiliśmy pierwszy krok, tworząc strukturę oceny, i przekonałeś się, że kluczem do sukcesu nie jest "magiczny prompt", ale precyzyjna definicja danych wejściowych i wyjściowych.

Zapamiętaj najważniejszą lekcję dotyczącą projektowania testów: odpowiedź finalna musi być binarna. Dążymy do jednoznacznego TAK lub NIE. Wprowadzanie skal czy ocen opisowych to zaproszenie do problemów z interpretacją i brakiem powtarzalności. Tylko twarde, binarne asercje pozwalają na automatyczne wyłapywanie regresji.

W kolejnych lekcjach weźmiemy ten fundament i zgłębimy nasz przykład. Zaczniemy badać, jak skalować takie leaderboardy, jak definiować bardziej zaawansowane metryki i jak łączyć to wszystko w jeden spójny system, który pozwoli Ci spać spokojnie, gdy Twój agent działa na produkcji.

## Przydatne linki

1. [The Leaderboard Illusion](https://arxiv.org/pdf/2504.20879)
2. [Pelican Arena Leaderboard](https://pelicans.borg.games)
3. [Investigating Data Contamination in Modern Benchmarks for Large Language Models](https://arxiv.org/pdf/2311.09783)
4. [What's going on with the Open LLM Leaderboard?](https://huggingface.co/blog/open-llm-leaderboard-mmlu)
5. [Changing Answer Order Can Decrease MMLU Accuracy](https://arxiv.org/pdf/2406.19470)
6. [Does SWE-Bench-Verified Test Agent Ability or Model Memory?](https://arxiv.org/abs/2512.10218)
7. [SWE-Bench+: Enhanced Coding Benchmark for LLMs](https://arxiv.org/pdf/2410.06992)